# SmolBench family-ladder study -- statistical analyses

One notebook for every statistic the study reports. It **imports the live
analysis modules** and calls them; it re-implements nothing that already has
a home in `notebooks/*/`. Three scripts are *ported* here, and only their
statistics are (sections 7-9): `scripts/posterior_power.py`,
`scripts/flip_probe.py --stage analyze`, and `scripts/flip_free_bound.py`.
Those three are archived in **`pr4_scripts_2026-08-25.zip`**; this notebook
is what replaces them.

## Rules this notebook runs under (user rulings, 2026-08-25)

1. **Archived data is accessed on S3, never written to a local path.**
   Sections 0, 5, 8 and 9 stream objects out of
   `s3://smolbench-results-414266451290/archives/2026-08-25/` straight into
   memory (`S3Archive` below, the same read/keys/sha256 logic as
   `tests/conftest.py::S3Archive`). Nothing here downloads the archive.
2. **`RUN_HEAVY` gates everything that needs the full results store**
   (`harness.sync_down()`, the `analysis/2026-08-16` snapshot, GPU, or
   Lean). Those cells are complete, runnable code -- they are switched off,
   not stubbed out.
3. **Saved with all outputs cleared.** Re-run it to reproduce the numbers.

Live AWS credentials are a *baseline* requirement: the ungated cells in
sections 0, 5, 8 and 9 read the archive. Everything else runs offline (or
skips).

## Section 0 -- setup and provenance

**What this is.** The anchor for everything below: the repository root, the
live analysis modules bound under unambiguous names, the archive handle, and
a live provenance listing (key + size + sha256) of every archived object this
notebook reads.

**Inputs.** The repo working tree (for the modules) and the S3 archive prefix
(for the evidence).

**Descends from.** `tests/tooling/test_analysis_stats.py` (the `_load`/`_bound`
import pattern, lines 71-85) and `tests/conftest.py::S3Archive` (the
streaming reader). Both legs ship a file called `power_analysis.py`, and each
module imports its siblings *by bare name* after putting its own directory on
`sys.path`; whichever leg imported first would otherwise own
`sys.modules["power_analysis"]` for the rest of the session. `_bound` binds
the right sibling for exactly the duration of each exec.

In [ ]:
"""Anchor the repo, then load every live analysis module under a unique name."""
import importlib.util
import json
import sys
from contextlib import contextmanager
from pathlib import Path


def find_repo(start: Path | None = None) -> Path:
    """Walk up from `start` (default: cwd) to the directory holding pyproject.toml.

    Notebooks have no ``__file__``, so the repo root is recovered from the
    working directory instead. This raises rather than guessing: a wrong
    root would silently point every path below at the wrong tree.
    """
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").is_file():
            return cand
    raise RuntimeError(
        f"no pyproject.toml at or above {here}: run this notebook from inside "
        "the SmolBench checkout"
    )


REPO = find_repo()
IND = REPO / "notebooks" / "induction"
DED = REPO / "notebooks" / "deduction"
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


def _load(name: str, path: Path):
    """Exec the module at `path` under `name`, registering it before exec."""
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module          # dataclass annotations resolve via sys.modules
    spec.loader.exec_module(module)
    return module


@contextmanager
def _bound(**modules):
    """Bind modules under bare names for the duration of the block."""
    saved = {n: sys.modules.get(n) for n in modules}
    sys.modules.update(modules)
    try:
        yield
    finally:
        for n, old in saved.items():
            if old is None:
                sys.modules.pop(n, None)
            else:
                sys.modules[n] = old


# --- deduction leg ---------------------------------------------------------
ded_pa = _load("ded_power_analysis", DED / "power_analysis.py")
with _bound(power_analysis=ded_pa):
    error_bars = _load("ded_error_bars", DED / "error_bars.py")
    with _bound(error_bars=error_bars):
        hint_vs_noise = _load("ded_hint_vs_noise", DED / "hint_vs_noise.py")

# --- induction leg ---------------------------------------------------------
ind_pa = _load("ind_power_analysis", IND / "power_analysis.py")
with _bound(power_analysis=ind_pa):
    paired = _load("ind_paired_analysis", IND / "paired_analysis.py")
    with _bound(paired_analysis=paired):
        significance = _load("ind_significance_report", IND / "significance_report.py")
        with _bound(significance_report=significance):
            extens_vs_noise = _load("ind_extens_vs_noise", IND / "extens_vs_noise.py")
response_audit = _load("ind_response_audit", IND / "response_audit.py")
run_study = _load("ind_run_study", IND / "run_study.py")

# ``notebooks/_power_common.py`` is imported by BOTH legs' power_analysis (each
# adds ``notebooks/`` to sys.path itself), so it is already in sys.modules;
# name it here so this notebook can cite its constants directly.
power_common = sys.modules["_power_common"]

print("repo          :", REPO)
print("interpreter   :", sys.executable)
print("induction lanes:", len(ind_pa.MODELS), " families:", len(ind_pa.FAMILIES))
print("deduction lanes:", len(ded_pa.MODELS), " families:", len(ded_pa.FAMILIES))
print("run_study roster:", len(run_study.MODELS), "lanes x", len(run_study.INFO_TYPES),
      "info arms, R =", run_study.N_REPLICATES)

In [ ]:
"""The heavy-work gate. Flip to True only with the results store in reach."""
#: Everything needing the FULL results store -- ``harness.sync_down()`` into
#: ``notebooks/induction/results``, the ``analysis/2026-08-16`` deduction
#: snapshot, GPU, or Lean -- is behind this flag. The archive-backed cells
#: (sections 0, 5's recovery summary, 8, 9) are NOT gated: they stream a few
#: small JSON objects and are the point of this notebook.
RUN_HEAVY = False

#: Analysis snapshot published by ``scripts/results/snapshot_analysis_data.py``. This
#: is the results STORE, not the 2026-08-25 archive. ``error_bars.py`` and
#: ``hint_vs_noise.py`` take a DIRECTORY of ``<model>/verified_rows.jsonl``,
#: so a heavy run has to materialise it locally (~GBs); see section 5.
SNAPSHOT_S3 = "s3://smolbench-results-414266451290/analysis/2026-08-16"
SNAPSHOT_REGION = "us-west-2"

#: Where a heavy run puts the snapshot. Scratch, never the repo tree.
import os
import tempfile

SCRATCH = Path(os.environ.get("SMOLBENCH_SCRATCH", tempfile.gettempdir())) / "smolbench_analysis"
ROWS_DIR = SCRATCH / "deduction_rows"

print(f"RUN_HEAVY = {RUN_HEAVY}")
print(f"scratch   = {SCRATCH}")

In [ ]:
"""Read-only, in-memory access to the 2026-08-25 archive prefix on S3.

Same read/keys/sha256 logic as ``tests/conftest.py::S3Archive`` (copied, not
imported: ``tests/`` is not an importable package). Nothing is written to
disk -- the user's ruling is that archived data is accessed on AWS.
"""
import hashlib
import posixpath

import boto3

ARCHIVE = "s3://smolbench-results-414266451290/archives/2026-08-25"
ARCHIVE_REGION = "us-west-2"


def parse_s3_uri(uri: str) -> tuple[str, str]:
    """Split ``s3://bucket/prefix`` into ``(bucket, prefix)``."""
    if not uri.startswith("s3://"):
        raise ValueError(f"not an s3 uri: {uri!r}")
    bucket, _, prefix = uri[len("s3://"):].partition("/")
    return bucket, prefix.strip("/")


class S3Archive:
    """Stream objects out of an ``s3://bucket/prefix`` archive root."""

    def __init__(self, uri: str, region: str | None = None) -> None:
        self.bucket, self.prefix = parse_s3_uri(uri)
        self._client = boto3.client("s3", region_name=region)

    def _key(self, rel: str) -> str:
        rel = posixpath.normpath(rel)
        return f"{self.prefix}/{rel}" if self.prefix else rel

    def keys(self, rel_prefix: str) -> list[str]:
        """List archive-relative paths under `rel_prefix` (a directory)."""
        full = self._key(rel_prefix).rstrip("/") + "/"
        out: list[str] = []
        paginator = self._client.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=self.bucket, Prefix=full):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                out.append(key[len(self.prefix) + 1:] if self.prefix else key)
        return out

    def exists(self, rel: str) -> bool:
        try:
            self._client.head_object(Bucket=self.bucket, Key=self._key(rel))
            return True
        except self._client.exceptions.ClientError:
            return False

    def open(self, rel: str):
        """Return a streaming body for one object (read it, do not save it)."""
        try:
            return self._client.get_object(Bucket=self.bucket, Key=self._key(rel))["Body"]
        except self._client.exceptions.NoSuchKey as exc:
            raise FileNotFoundError(self._key(rel)) from exc

    def read(self, rel: str) -> bytes:
        return self.open(rel).read()

    def text(self, rel: str) -> str:
        return self.read(rel).decode("utf-8", errors="replace")

    def json(self, rel: str):
        return json.loads(self.text(rel))

    def size(self, rel: str) -> int:
        return int(self._client.head_object(Bucket=self.bucket, Key=self._key(rel))["ContentLength"])

    def sha256(self, rel: str) -> str:
        h = hashlib.sha256()
        for chunk in self.open(rel).iter_chunks(1 << 20):
            h.update(chunk)
        return h.hexdigest()


archive = S3Archive(ARCHIVE, ARCHIVE_REGION)
print("archive root:", ARCHIVE)

In [ ]:
"""Provenance: every archived object this notebook reads, with its sha256.

Executed live, on purpose. A number in sections 8 and 9 is only as good as
the bytes it came from, and this cell is the record of exactly which bytes
those were.
"""
#: archive-relative key -> which section consumes it.
ARCHIVE_INPUTS = {
    "notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/flip_report.json":
        "section 8 -- score-level flip rate (DETERMINISM_PLAN 6.2)",
    "notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/sample_manifest.json":
        "section 8 -- sample provenance (whitelist sha256, population size)",
    "notebooks/deduction/results/flip_free_bound_2026-08-18.json":
        "section 9 -- free flip bound (DETERMINISM_PLAN 6.3)",
    "notebooks/deduction/results/dojoinit_recovery_2026-08-18/report.json":
        "section 5 -- DojoInit recovery, sensitivity pool for error_bars",
}

print(f"{'sha256':>16} {'bytes':>10}  key")
print("-" * 100)
provenance = {}
for rel, use in ARCHIVE_INPUTS.items():
    digest, nbytes = archive.sha256(rel), archive.size(rel)
    provenance[rel] = {"sha256": digest, "bytes": nbytes, "used_by": use}
    print(f"{digest[:16]} {nbytes:>10}  {rel}")
    print(f"{'':>16} {'':>10}  -> {use}")
print("-" * 100)
print(f"{len(provenance)} archived object(s) resolved under {ARCHIVE}")

## Section 1 -- induction sizing (prospective)

**What this is.** The pre-registered replicate-sizing analysis for the
induction leg: per-family omnibus gates, the 210-contrast PRIMARY tier at
`ALPHA/210`, the 63-contrast SECONDARY tier under BH, and the recommended
`R`. It is **prospective only** -- it reads the *pilot* seed
(`PILOT_SEED = 0`) and deliberately refuses to read the collected block, so
sizing never becomes circular. The posterior counterpart is section 7.

**Inputs.** `notebooks/induction/results/<lane>_<arm>/rep_0.yaml`, which
`InductionExperiment.harness.sync_down()` writes from the S3 results store.

**Descends from.** The live module `notebooks/induction/power_analysis.py`
(imported above, *not* re-implemented) and the archived record
`notebooks/induction/POWER_ANALYSIS_2026-08-14.md`
(`pr4_notebook_records_2026-08-25.zip`).

In [ ]:
"""Induction replicate sizing. Needs the pilot replicate of every lane."""
if RUN_HEAVY:
    # ind_pa.RESULTS_DIR is anchored on the module's own __file__, so this
    # is notebooks/induction/results regardless of the notebook's cwd.
    print("results dir:", ind_pa.RESULTS_DIR)
    print("alpha primary:", ind_pa.ALPHA_PRIMARY, " alpha secondary:", ind_pa.ALPHA_SECONDARY)
    ind_pa.main()
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results "
          "-- InductionExperiment.harness.sync_down() first")

## Section 2 -- induction paired re-analysis

**What this is.** The correction that produced the published induction
headline. The pre-registered test was an *unpaired* CMH, but both arms of a
contrast are drawn from the same replicate seeds, and a seed fixes the label
alphabet and answer vector shared by its 9 harmonics. So the notebook reports
three tests over the same 210 contrasts:

* `mcnemar_exact_p` -- exact conditional McNemar on the paired discordance;
* `signflip_exact_p` -- the **seed-level** exact sign-flip randomisation test,
  which treats the replicate (not the mark) as the exchangeable unit. This is
  the cluster-corrected primary; its resolution floor is `2 / 2**30` at R=30;
* `cmh_unpaired_p` -- the original statistic, kept so the paired-vs-unpaired
  difference isolates the *pairing* and nothing else;

plus `holm` (FWER over the 210-contrast family) and `design_effect`, the
observed/independence-assumed variance ratio of the per-seed arm difference:
the quantity CMH's denominator omits.

**Inputs.** `load_marks()` over every landed `rep_<seed>.yaml`.

**Descends from.** `notebooks/induction/paired_analysis.py` and the archived
record `notebooks/induction/PAIRED_ANALYSIS_RESULTS.md`
(`pr4_notebook_records_2026-08-25.zip`).

In [ ]:
"""Per-contrast paired statistics, built from paired_analysis' own primitives."""
if RUN_HEAVY:
    import numpy as np

    correct, valid = paired.load_marks()
    contrasts = ind_pa.build_primary_contrasts()
    assert len(contrasts) == ind_pa.N_PRIMARY

    rows = []
    for label, key_a, key_b in contrasts:
        a, b, seed_idx = paired.aligned(correct, valid, key_a, key_b, drop_invalid=False)
        disc_b = int((a & ~b).sum())
        disc_c = int((~a & b).sum())
        rows.append({
            "label": label,
            "n_items": int(a.size),
            "n_seeds": int(np.unique(seed_idx).size),
            "acc_a": float(a.mean()),
            "acc_b": float(b.mean()),
            "b": disc_b,
            "c": disc_c,
            "p_mcnemar": paired.mcnemar_exact_p(disc_b, disc_c),
            "p_signflip": paired.signflip_exact_p(paired.seed_diffs(a, b, seed_idx)),
            "p_cmh": paired.cmh_unpaired_p(a, b, seed_idx),
            "deff": paired.design_effect(a, b, seed_idx),
        })

    for stat in ("p_signflip", "p_mcnemar", "p_cmh"):
        rej = paired.holm(np.array([r[stat] for r in rows]), ind_pa.ALPHA)
        print(f"Holm rejections over {len(rows)} PRIMARY contrasts, {stat:>10}: "
              f"{int(rej.sum())}")

    deffs = np.array([r["deff"] for r in rows if r["deff"] is not None])
    print(f"design effect over {deffs.size} measurable contrasts: "
          f"median {np.median(deffs):.2f}, IQR "
          f"[{np.percentile(deffs, 25):.2f}, {np.percentile(deffs, 75):.2f}], "
          f"max {deffs.max():.2f}")
    print(f"(>1 means the unpaired CMH denominator is too small, i.e. "
          f"anticonservative)")
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

In [ ]:
"""The canonical paired report: both invalid-handling passes, plus SECONDARY BH."""
if RUN_HEAVY:
    paired.main()
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

## Section 3 -- induction significance report and the selection-rule audits

**What this is.** Five things, in the order the study needed them.

1. `significance_report.main()` -- the published induction headline. Hochberg
   at FWER 0.05 over the 210 PRIMARY contrasts, with the **collapse census**
   attached: a lane whose arm degenerated into repetition is reported as a
   first-class result with a mechanism annotation, never quarantined (user
   ruling, `no-quarantine-collapse-is-a-result`).
2. `extens_vs_noise.main()` -- the information-vs-length contrast, with the
   `mechanism()` classifier that separates a genuine information effect from
   a length/compliance artefact.
3. `response_audit.main()` -- response-level audit (empty rate, scored rate,
   answer-in-response, longest run) behind the collapse annotations.
4. `verify_survivorship` -- the empty-response profile of ministral-3-14b's
   re-collected seeds against the rest: the gap the lane's
   missing-not-at-random caveat rests on.
5. `build_selected_tree` -> `compare_selection_rules` -> `check_currency` --
   the EARLIEST-vs-NEWEST audit. Earliest-wins is the user's ruling; this
   chain measures what it changed (140 multi-attempt cells) and re-gates the
   local tree **on content**, not on row counts.

**Inputs.** `notebooks/induction/results` for 1-4; for 5, an
`aws s3 ls --recursive` key listing of `analysis/2026-08-16/induction` plus
two full results trees (`ind_earliest`, `ind_newest`).

**Descends from.** The live modules `significance_report.py`,
`extens_vs_noise.py`, `response_audit.py`, `verify_survivorship.py`,
`compare_selection_rules.py`, `check_currency.py`, `build_selected_tree.py`,
and the archived records `notebooks/induction/CONFOUND_AUDIT_2026-08-13.md`
and `MULTIPLICITY_PLAN.md` (`pr4_notebook_records_2026-08-25.zip`).

> **The last four are top-level scripts, not importable modules**: they run
> their analysis at import time and hardcode absolute paths, including a
> *session-specific* scratch directory
> (`build_selected_tree.py:25`, `compare_selection_rules.py:15`) that belongs
> to the 2026-08-16 session and no longer exists. They are therefore driven
> with `runpy.run_path` under an explicit precondition check, which names
> exactly what is missing instead of failing halfway. `ind_newest` (the
> pre-ruling tree) is **not reproducible** by `build_selected_tree`, which
> only ever builds `ind_earliest`; without an `ind_newest` tree the
> selection-rule comparison cannot be re-run at all, and the audit's
> conclusion stands on its archived record.

In [ ]:
"""Published induction significance report, plus the two arm-level audits."""
if RUN_HEAVY:
    significance.main()
    print("\n" + "=" * 100 + "\n")
    extens_vs_noise.main()
    print("\n" + "=" * 100 + "\n")
    response_audit.main()
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

In [ ]:
"""The selection-rule / survivorship audits: top-level scripts, driven by runpy."""
if RUN_HEAVY:
    import runpy
    import subprocess

    # 5a. The missing-not-at-random profile. Reads the repo results tree only.
    runpy.run_path(str(IND / "verify_survivorship.py"), run_name="__main__")

    # 5b. Key listing of the analysis snapshot's induction leg: "<size> <key>"
    # per line, which is what both selection-rule scripts parse. This lists
    # keys; it downloads no objects.
    scratch_of_scripts = Path(
        "/tmp/claude-1001/-workspace-SmolBench/"
        "54dbdffb-0485-4ced-9231-fa52049df286/scratchpad"
    )  # hardcoded at build_selected_tree.py:25 / compare_selection_rules.py:15
    scratch_of_scripts.mkdir(parents=True, exist_ok=True)
    keys_file = scratch_of_scripts / "ind_keys.txt"
    bucket, prefix = parse_s3_uri(SNAPSHOT_S3)
    listing = subprocess.run(
        ["aws", "s3", "ls", f"s3://{bucket}/{prefix}/induction/", "--recursive",
         "--region", SNAPSHOT_REGION],
        check=True, capture_output=True, text=True).stdout
    # `aws s3 ls --recursive` prints "<date> <time> <size> <key>"; both
    # scripts want "<size> <key>".
    keys_file.write_text("".join(
        f"{parts[2]} {parts[3]}\n"
        for parts in (line.split(None, 3) for line in listing.splitlines() if line.strip())
    ))
    print(f"wrote {keys_file} ({len(keys_file.read_text().splitlines())} objects)")

    # 5c. Re-gate the local tree on CONTENT (size per earliest-wins cell).
    # check_currency.py reads its key listing from sys.argv[1] at import time.
    import sys as _sys
    _argv = _sys.argv
    try:
        _sys.argv = ["check_currency.py", str(keys_file)]
        runpy.run_path(str(IND / "check_currency.py"), run_name="__main__")
    finally:
        _sys.argv = _argv

    # 5d. EARLIEST-vs-NEWEST. build_selected_tree materialises ind_earliest;
    # ind_newest is the PRE-RULING tree and cannot be rebuilt by any script in
    # this repo, so refuse loudly rather than compare a tree against itself.
    runpy.run_path(str(IND / "build_selected_tree.py"), run_name="__main__")
    newest = scratch_of_scripts / "ind_newest"
    if not newest.is_dir():
        raise SystemExit(
            f"{newest} is absent. compare_selection_rules.py compares the "
            "EARLIEST-selected tree against the pre-ruling NEWEST-selected "
            "tree; build_selected_tree.py only builds the former. The NEWEST "
            "tree was the repo working tree as of 2026-08-16 and is not "
            "reproducible from S3 alone -- the comparison's result stands on "
            "its archived record, not on a re-run."
        )
    runpy.run_path(str(IND / "compare_selection_rules.py"), run_name="__main__")
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results, an S3 "
          "key listing, and both selection-rule trees")

## Section 4 -- deduction sizing (prospective)

**What this is.** The deduction leg's replicate-sizing analysis: for each of
the 21 within-family PRIMARY contrasts (and 63 cross-family SECONDARY ones),
a block bootstrap over **theorem blocks** gives the `n_theorems` power curve,
and a Beta-mixture projection gives the replicate sizing. Blocking is the
point: cells of one theorem are not independent draws, and `power_analysis`'s
own block bootstrap is this study's answer to that everywhere it matters.

**Inputs.** `<results-dir>/runs/scaling_<model>/verified_rows.jsonl` for all
21 lanes, or `--s3` to pull them from `deduction/runs/` into a temp dir.

**Descends from.** `notebooks/deduction/power_analysis.py`. Note that
`power_analysis.py` is **not** what produced the published deduction
headline -- `error_bars.py` (section 5) is.

In [ ]:
"""Deduction replicate sizing over the 21 lanes."""
if RUN_HEAVY:
    print("primary alpha:", ded_pa.ALPHA_PRIMARY,
          " secondary alpha:", ded_pa.ALPHA_SECONDARY)
    # --s3 pulls verified_rows.jsonl for all 21 lanes into a temp dir it owns.
    # Swap for ["--results-dir", str(<dir>)] to analyse a local tree instead.
    rc = ded_pa.main(["--s3", "--sims", str(ded_pa.SIMS)])
    print("exit code:", rc)
else:
    print("skipped (RUN_HEAVY=False): needs the deduction run files "
          "(21 lanes x verified_rows.jsonl)")

## Section 5 -- deduction error bars (the published headline)

**What this is.** The statistic behind the study's published deduction
numbers. `build_pool` assembles the paired 21-way pool under one explicit
denominator rule (`count_as_failure=True`: a model-dependent no-survivor cell
scores 0 rather than dropping out); `block_matrix` reduces it to
`(n_theorems, n_models)` successes with a per-theorem cell count. Then:

* `bootstrap_stats` -- block bootstrap over theorem blocks with **BCa**
  intervals per lane (`B` up to 500k; a jackknife over blocks supplies the
  acceleration);
* `diff_ci` -- the paired difference interval, differenced *inside* each
  resample so the shared theorem draw cancels;
* `paired_mcnemar` -- cell-level exact McNemar for the same contrast;
* `block_signflip_p` -- the cluster-corrected primary: per-theorem
  differences with exchangeable signs, `(#{|perm| >= |obs|} + 1) / (B + 1)`;
* `holm` -- FWER over the 21 PRIMARY contrasts;
* `mode_report` -- the full table, plus the design effect against a naive
  binomial and the sensitivity pools under the other denominator rules.

**Inputs.** A **directory** of `<model>/verified_rows.jsonl` (`--rows-dir`)
and, for the sensitivity arm, a **directory** of
`<model>/recovered_rows.jsonl` (`--recovery-dir`).

> **Both are directory paths, and that is a real constraint here.**
> `build_pool` opens files off the filesystem; it has no streaming entry
> point. The rows come from the `analysis/2026-08-16` snapshot, which a heavy
> run syncs to scratch (gigabytes -- gated). The recovery directory lives in
> the **2026-08-25 archive**, and the ruling forbids writing archived data to
> a local path, so this notebook does **not** materialise it and the
> post-recovery sensitivity arm is left out of the gated run. What the
> notebook does instead, ungated, is stream the recovery's own
> `report.json` out of the archive and summarise it.

**Descends from.** `notebooks/deduction/error_bars.py`, the archived evidence
tree `notebooks/deduction/results/dojoinit_recovery_2026-08-18/`
(`pr4_evidence_and_data_2026-08-25.zip`, on S3 at the archive prefix), and
the archived records `notebooks/DOJOINIT_RECOVERY_2026-08-18.md` and
`FAMILY_LADDER_ANALYSIS_2026-08-16.md`
(`pr4_notebook_records_2026-08-25.zip`).

In [ ]:
"""Stream the DojoInit recovery report out of the archive (no download)."""
REC_REPORT = "notebooks/deduction/results/dojoinit_recovery_2026-08-18/report.json"
rec = archive.json(REC_REPORT)

print(f"{REC_REPORT}")
print(f"  sha256 {provenance[REC_REPORT]['sha256']}")


def _summarise(obj, indent=2, path=""):
    """Print a shallow, type-aware summary of a nested JSON report."""
    pad = " " * indent
    if isinstance(obj, dict):
        for key, val in obj.items():
            if isinstance(val, dict):
                print(f"{pad}{key}:")
                _summarise(val, indent + 2, f"{path}/{key}")
            elif isinstance(val, list):
                print(f"{pad}{key}: list[{len(val)}]")
            elif isinstance(val, str) and len(val) > 88:
                print(f"{pad}{key}: {val[:88]}...")
            else:
                print(f"{pad}{key}: {val}")


_summarise(rec)
print("\nThe post-recovery SENSITIVITY pool is NOT computed below: error_bars'")
print("--recovery-dir needs a local directory of <model>/recovered_rows.jsonl,")
print("and the recovery tree is archived data (accessed on S3, never downloaded).")

In [ ]:
"""The published deduction error bars. Needs a LOCAL rows directory."""
if RUN_HEAVY:
    import subprocess

    import numpy as np

    # Materialise the analysis snapshot's deduction leg. This is the results
    # STORE (analysis/2026-08-16), not the 2026-08-25 archive: gigabytes, and
    # the reason this cell is gated.
    ROWS_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["aws", "s3", "sync", f"{SNAPSHOT_S3}/deduction/", str(ROWS_DIR),
         "--exclude", "*", "--include", "*/verified_rows.jsonl",
         "--region", SNAPSHOT_REGION, "--only-show-errors"],
        check=True)

    models, blocks, rungs, meta = error_bars.build_pool(
        ROWS_DIR, recovery_dir=None, count_as_failure=True)
    succ, size = error_bars.block_matrix(models, blocks)
    per_lane = dict(meta["own_rate"])
    print(f"pool: {succ.shape[0]} theorem blocks, {int(size.sum())} cells, "
          f"{len(models)} lanes")

    # The pieces, called directly, before the full report prints them.
    bs = error_bars.bootstrap_stats(succ, size, B=20_000,
                                    seed=error_bars.SIGNFLIP_SEED, alpha=0.05)
    contrasts = ded_pa.build_within_family_contrasts()
    p_signflip = error_bars.block_signflip_p(succ, models, contrasts)
    rej = error_bars.holm(p_signflip, error_bars.ALPHA)
    print(f"PRIMARY contrasts rejected under Holm (block sign-flip): "
          f"{int(rej.sum())} / {len(contrasts)}")
    # build_within_family_contrasts() yields (label, model_a, model_b) triples.
    for (label, a, b), p, r in list(zip(contrasts, p_signflip, rej))[:3]:
        d = error_bars.diff_ci(bs, models.index(a), models.index(b))
        nb, nc, p_mc = error_bars.paired_mcnemar(models, blocks, a, b)[:3]
        print(f"  {label}: diff {d['diff']:+.4f} "
              f"BCa [{d['lo']:+.4f}, {d['hi']:+.4f}]"
              f"{' (percentile fallback)' if d['fallback'] else ''}  "
              f"signflip p={p:.2e}  McNemar b/c={nb}/{nc} p={p_mc:.2e}  "
              f"Holm={'yes' if r else '.'}")

    # The full published report. --recovery-dir is deliberately absent: see
    # this section's note. mode_report then states that post-recovery pools
    # are not shown.
    sensitivity = []
    for caf in (True, False):
        if caf is True:
            continue
        _m, _b, _r, _meta = error_bars.build_pool(ROWS_DIR, recovery_dir=None,
                                                  count_as_failure=caf)
        _succ, _size = error_bars.block_matrix(_m, _b)
        _p = error_bars.block_signflip_p(_succ, _m, ded_pa.build_within_family_contrasts())
        _paired = _succ.sum(axis=0) / _size.sum()
        _gap = max(abs(_paired[_m.index(mm)] - _meta["own_rate"][mm])
                   for mm in ded_pa.MODELS)
        sensitivity.append(("drop no-survivor (Mathlib only)", int(_size.sum()),
                            _succ.shape[0], int(error_bars.holm(_p).sum()), _gap))
    error_bars.mode_report(succ, size, models, blocks, per_lane, 20_000, None,
                           meta=meta, sensitivity=sensitivity)
else:
    print("skipped (RUN_HEAVY=False): needs a local <model>/verified_rows.jsonl "
          "tree synced from the analysis/2026-08-16 snapshot")

## Section 6 -- deduction hint vs noise

**What this is.** The deduction leg's information-vs-length contrast.
`hint:3` and `noise:3` are byte-identical except for `hint:3`'s trailing
1-hop transitive premise-closure block, which `noise:3` replaces with
token-matched padding. So this tests *supplementary* background on top of an
already-complete direct-premise context -- **not** the same manipulation as
the induction `extens`-vs-`noise` contrast, and the report says so. One cell
per theorem per model, so exact McNemar applies with no cluster correction,
and Holm runs over the 21 models. The report also states the minimum
detectable effect, so a null is not read as an absence of measurement.

**Inputs.** The same `--rows-dir` directory of `<model>/verified_rows.jsonl`
as section 5.

**Descends from.** `notebooks/deduction/hint_vs_noise.py`.

In [ ]:
"""hint:3 vs noise:3, per model, paired within theorem."""
if RUN_HEAVY:
    rc = hint_vs_noise.main(["--rows-dir", str(ROWS_DIR)])
    print("exit code:", rc)
else:
    print("skipped (RUN_HEAVY=False): needs the same rows directory as section 5")

## Section 7 -- posterior power (port of `scripts/posterior_power.py`)

**What this is.** The *posterior* counterpart to sections 1 and 4: at the R
already collected, which planned contrasts are settled and which would
actually benefit from more data.

It is deliberately **not** "observed power". Observed power is a monotone
function of the p-value, so it carries no information beyond it: a
non-significant result always yields low observed power, and reporting it
dresses "p was large" up as independent evidence. Instead every contrast
lands in one of three states:

| state | rule | meaning |
|---|---|---|
| `DECIDED` | test rejects at the corrected alpha | settled; more replicates cannot unsettle it |
| `EQUIVALENT` | not significant **and** the CI lies entirely inside +-MEI | a demonstrated near-tie; also settled |
| `UNDECIDED` | not significant **and** the CI still spans MEI | the only state where more data helps |

**MEI is pre-specified**, not observed: a claim that a difference is too
small to care about must say in advance how big "care about" is. The
equivalence interval is a `1 - 2*alpha` interval (the TOST convention: such
an interval inside +-MEI is exactly two one-sided tests both rejecting at
alpha).

`MIN_R_FOR_EQUIVALENCE = 5` guards the one way this can lie: a bootstrap over
2 agreeing replicates has near-zero width, fits inside any MEI, and
manufactures an equivalence claim out of almost no data. Below the threshold
the honest state is `UNDECIDED`.

**What was ported, and what was dropped.** Ported: the classifier, the MEI
framing, and the `MIN_R_FOR_EQUIVALENCE` guard
(`scripts/posterior_power.py:88-98` and `:342-453`). Dropped in favour of the
live modules -- the script carried private copies of all three:

* its `cmh`/`chi2_sf_1df` -> `paired_analysis.cmh_unpaired_p` (same
  continuity correction and hypergeometric variance, driven off the real
  item-matched arrays);
* its `boot_ci` -> `error_bars.bootstrap_stats` + `error_bars.diff_ci` (BCa
  instead of percentile, and **paired** across the shared replicate draw
  rather than resampling the two arms independently);
* its `replicates_needed` -> `power_analysis.replicates_needed` (which
  returns `(needed, curve)` over `POWER_TARGETS` and takes an `alpha`, so the
  call site adapts rather than the module).

Its `load_study`/`invalid_counts` regex loaders are dropped too:
`paired_analysis.load_marks` already reads the same files, and returns the
validity mask that keeps invalid marks distinguishable from wrong ones.

**The family changes with the roster.** The script was written for the
retired 3-model x 4-info pilot, giving `N_TESTS = 30` and
`ALPHA = 0.05/30`. Parameterised by the *current* 21-lane roster from
`notebooks/induction/run_study.py`, the same all-pairs construction gives
`4*C(21,2) + 21*C(4,2) = 966` tests and a far stricter alpha. That family is
also **not** the study's pre-registered one: the study registers 210 PRIMARY
within-family ladder contrasts at `ALPHA/210` (`power_analysis.N_PRIMARY`).
Both are computed below, and the report states which alpha it used.

**Inputs.** `notebooks/induction/results` (gated). The classifier itself is
pure and is exercised on synthetic counts, ungated.

**Descends from.** `scripts/posterior_power.py`, archived in
**`pr4_scripts_2026-08-25.zip`**.

In [ ]:
"""Posterior-power port: the classifier, the MEI framing, and the R guard."""
from itertools import combinations

import numpy as np

#: Minimum replicates per side before an EQUIVALENT verdict is allowed
#: (posterior_power.py:90-97). Equivalence is a positive claim: it needs
#: enough replicates to have been able to refute itself.
MIN_R_FOR_EQUIVALENCE = 5

#: Default pre-specified minimum effect of interest, absolute accuracy.
#: Pre-specify it; do not tune it to the data.
DEFAULT_MEI = 0.05


def posterior_family(models, infos) -> int:
    """Count the all-pairs contrast family: model-within-info plus info-within-model.

    This includes the chance-floor ``zero`` contrasts on purpose. They are
    trivially significant, but dropping them from the correction after
    seeing the data is exactly the multiplicity abuse the correction
    exists to prevent.
    """
    return len(infos) * len(list(combinations(models, 2))) + \
        len(models) * len(list(combinations(infos, 2)))


def build_posterior_contrasts(models, infos):
    """Build every (label, key_a, key_b) in the all-pairs posterior family."""
    out = []
    for info in infos:
        for m_a, m_b in combinations(models, 2):
            out.append((f"[{info}] {m_a} vs {m_b}", (m_a, info), (m_b, info)))
    for model in models:
        for i_a, i_b in combinations(infos, 2):
            out.append((f"[{model}] {i_a} vs {i_b}", (model, i_a), (model, i_b)))
    return out


def classify(p, ci_lo, ci_hi, mei, r_min, alpha,
             min_r=MIN_R_FOR_EQUIVALENCE) -> str:
    """Sort one contrast into DECIDED / EQUIVALENT / UNDECIDED.

    Parameters
    ----------
    p : float
        The contrast's test p-value (here: `paired_analysis.cmh_unpaired_p`).
    ci_lo, ci_hi : float
        Bounds of the ``1 - 2*alpha`` interval for the accuracy difference.
    mei : float
        Pre-specified minimum effect of interest, absolute accuracy.
    r_min : int
        Replicates on the SMALLER side of the contrast.
    alpha : float
        Multiplicity-corrected significance threshold.
    min_r : int, default MIN_R_FOR_EQUIVALENCE
        Replicates required before EQUIVALENT is allowed.

    Returns
    -------
    str
        ``"DECIDED"``, ``"EQUIVALENT"`` or ``"UNDECIDED"``. A contrast that
        would be EQUIVALENT on interval width alone, but has fewer than
        `min_r` replicates, is UNDECIDED: that is the honest state, not a
        near-tie.
    """
    if p < alpha:
        return "DECIDED"
    if ci_lo > -mei and ci_hi < mei and r_min >= min_r:
        return "EQUIVALENT"
    return "UNDECIDED"


def paired_diff_ci(a, b, seed_idx, alpha, n_boot=4000, seed=0) -> dict:
    """Bootstrap the accuracy difference ``a - b``, resampling REPLICATES.

    The replicate is the independent unit; harmonics inside one are strata
    of differing difficulty, not exchangeable draws, so they are never
    resampled. This delegates to `error_bars.bootstrap_stats`, treating
    each replicate as a "block" and the two arms as two "models", so the
    difference is taken INSIDE each resample and the shared replicate draw
    cancels (which the archived script's independent two-arm resampling
    did not do).

    Returns `error_bars.diff_ci`'s dict: ``diff`` (= mean(a) - mean(b)),
    ``lo``, ``hi``, ``se``, ``fallback``.
    """
    seeds = np.unique(seed_idx)
    succ = np.array([[a[seed_idx == s].sum(), b[seed_idx == s].sum()] for s in seeds],
                    dtype=float)
    size = np.array([(seed_idx == s).sum() for s in seeds], dtype=float)
    bs = error_bars.bootstrap_stats(succ, size, B=n_boot, seed=seed, alpha=alpha)
    # diff_ci(bs, ja, jb) is rate(jb) - rate(ja); column 0 is arm a, so
    # (ja=1, jb=0) gives a - b.
    return error_bars.diff_ci(bs, 1, 0)


# --- the current roster, and the two alphas it implies ----------------------
# run_study.MODELS maps SPEC KEY ("qwen3.5-27b") -> LOCAL RESULTS TAG
# ("qwen35_27b"). The analysis modules are keyed by the TAG -- load_marks
# reads RESULTS_DIR / f"{tag}_{info}" -- so the roster is taken through the
# mapping, exactly as build_selected_tree.py:31 does. The assert below pins
# the two name spaces together: a roster edit on one side that does not land
# on the other must fail loudly here, not silently skip every contrast.
ROSTER_SPECS = tuple(run_study.MODELS)                  # declaration order
ROSTER_MODELS = tuple(run_study.MODELS[k] for k in ROSTER_SPECS)   # local tags
ROSTER_INFOS = tuple(run_study.INFO_TYPES)       # intens / extens / noise_intens / zero
assert set(ROSTER_MODELS) == set(ind_pa.MODELS), (
    "run_study's local result tags and power_analysis.MODELS disagree: "
    f"{sorted(set(ROSTER_MODELS) ^ set(ind_pa.MODELS))}")
N_TESTS = posterior_family(ROSTER_MODELS, ROSTER_INFOS)
ALPHA_POSTERIOR = power_common.ALPHA / N_TESTS

print(f"roster: {len(ROSTER_MODELS)} lanes x {len(ROSTER_INFOS)} info arms "
      f"(notebooks/induction/run_study.py)")
print(f"  spec key -> results tag, e.g. {ROSTER_SPECS[0]} -> {ROSTER_MODELS[0]}"
      f"  (matches power_analysis.MODELS)")
print(f"all-pairs posterior family : N_TESTS = {N_TESTS}, "
      f"alpha = {power_common.ALPHA}/{N_TESTS} = {ALPHA_POSTERIOR:.3e}")
print(f"  (the archived script's retired 3-model roster gave "
      f"N_TESTS = {posterior_family(('gptoss', 'nemotron3', 'qwen35'), ROSTER_INFOS)})")
print(f"pre-registered PRIMARY family: N = {ind_pa.N_PRIMARY}, "
      f"alpha = {ind_pa.ALPHA_PRIMARY:.3e} -- a DIFFERENT family "
      f"(within-family ladder contrasts only)")
print(f"MIN_R_FOR_EQUIVALENCE = {MIN_R_FOR_EQUIVALENCE}, default MEI = {DEFAULT_MEI}")

In [ ]:
"""Self-test: exercise the ported classifier on synthetic counts. Ungated."""
import numpy as np

_MEI, _ALPHA = 0.05, 0.05

# 1. The pure decision table, including the guard branch that the real data
#    can never reach: every lane collected R = 30, so r_min < 5 never occurs
#    outside this test.
CASES = [
    # (p, lo, hi, r_min, expected, what it pins)
    (1e-9, +0.20, +0.40, 30, "DECIDED", "rejects: settled whatever the CI"),
    (1e-9, -0.01, +0.01, 3, "DECIDED", "rejection outranks the R guard"),
    (0.42, -0.02, +0.03, 30, "EQUIVALENT", "CI inside +-MEI, enough replicates"),
    (0.42, -0.02, +0.03, 3, "UNDECIDED", "same CI, too few replicates: the guard"),
    (0.42, -0.02, +0.03, 5, "EQUIVALENT", "exactly at the guard threshold"),
    (0.42, -0.09, +0.02, 30, "UNDECIDED", "CI still spans -MEI"),
    (0.42, -0.02, +0.09, 30, "UNDECIDED", "CI still spans +MEI"),
    (0.42, -0.05, +0.05, 30, "UNDECIDED", "CI touching +-MEI is NOT equivalence"),
]
for p, lo, hi, r_min, expected, why in CASES:
    got = classify(p, lo, hi, _MEI, r_min, _ALPHA)
    assert got == expected, f"classify({p}, {lo}, {hi}, r_min={r_min}) = {got} != {expected}"
    print(f"  ok  {got:<10} r_min={r_min:<3} CI [{lo:+.2f},{hi:+.2f}] p={p:<8.3g} {why}")

# 2. End to end on synthetic marks, through the LIVE modules the port reuses:
#    paired_analysis.cmh_unpaired_p for p, error_bars.bootstrap_stats/diff_ci
#    for the interval.
rng = np.random.default_rng(0)
N_HARM = ind_pa.N_HARMONICS


def synth(rate, n_seeds, gen):
    """Draw an (n_seeds * N_HARM) flat mark vector plus its replicate index."""
    marks = (gen.random((n_seeds, N_HARM)) < rate)
    seed_idx = np.repeat(np.arange(n_seeds), N_HARM)
    return marks.reshape(-1), seed_idx


def posterior_state(a, b, seed_idx, mei, alpha, n_seeds):
    """Run one synthetic contrast through the whole ported pipeline."""
    p = paired.cmh_unpaired_p(a, b, seed_idx)
    ci = paired_diff_ci(a, b, seed_idx, alpha=2 * alpha, n_boot=4000, seed=0)
    return p, ci, classify(p, ci["lo"], ci["hi"], mei, n_seeds, alpha)


# 2a. A large true effect -> DECIDED.
a, si = synth(0.95, 12, rng)
b, _ = synth(0.15, 12, rng)
p, ci, state = posterior_state(a, b, si, _MEI, _ALPHA, 12)
print(f"\n  synthetic 0.95 vs 0.15, R=12: diff {ci['diff']:+.3f} "
      f"CI [{ci['lo']:+.3f}, {ci['hi']:+.3f}] p={p:.2e} -> {state}")
assert state == "DECIDED", state

# 2b. A true null at a generous MEI with enough replicates -> EQUIVALENT.
a, si = synth(0.50, 40, rng)
b, _ = synth(0.50, 40, rng)
p, ci, state = posterior_state(a, b, si, 0.15, _ALPHA, 40)
print(f"  synthetic 0.50 vs 0.50, R=40, MEI=0.15: diff {ci['diff']:+.3f} "
      f"CI [{ci['lo']:+.3f}, {ci['hi']:+.3f}] p={p:.3f} -> {state}")
assert state == "EQUIVALENT", state

# 2c. The SAME null on 3 replicates -> UNDECIDED, via the guard alone.
a3, si3 = synth(0.50, 3, rng)
b3, _ = synth(0.50, 3, rng)
p3, ci3, state3 = posterior_state(a3, b3, si3, 0.30, _ALPHA, 3)
print(f"  synthetic 0.50 vs 0.50, R=3,  MEI=0.30: diff {ci3['diff']:+.3f} "
      f"CI [{ci3['lo']:+.3f}, {ci3['hi']:+.3f}] p={p3:.3f} -> {state3} "
      f"(MIN_R_FOR_EQUIVALENCE={MIN_R_FOR_EQUIVALENCE})")
assert state3 == "UNDECIDED", state3

print("\nposterior-power port: self-test PASSED "
      f"({len(CASES)} decision cases + 3 end-to-end synthetic contrasts)")

In [ ]:
"""Posterior power over the collected block. Needs the induction results tree."""
if RUN_HEAVY:
    import numpy as np

    MEI = DEFAULT_MEI
    ALPHA_USED = ALPHA_POSTERIOR      # all-pairs family; see this section's note

    correct, valid = paired.load_marks()
    # Gate on CONTENT: a roster/name-space mismatch must stop the section, not
    # quietly print a table of zeros.
    missing = [key for _, key_a, key_b in
               build_posterior_contrasts(ROSTER_MODELS, ROSTER_INFOS)
               for key in (key_a, key_b) if key not in correct]
    if missing:
        raise SystemExit(f"{len(set(missing))} roster condition(s) absent from "
                         f"load_marks(), e.g. {sorted(set(missing))[:3]}")
    invalid_counts = {
        key: int(sum((~v).sum() for v in valid[key].values())) for key in valid
    }

    print(f"MEI {MEI:.3f} absolute accuracy   alpha {power_common.ALPHA}/{N_TESTS} "
          f"= {ALPHA_USED:.2e}")
    print(f"\n{'contrast':>46} {'diff':>8} {'1-2a CI':>20} {'p (CMH)':>10}  state")
    print("-" * 104)

    tally = {"DECIDED": 0, "EQUIVALENT": 0, "UNDECIDED": 0, "SKIPPED": 0}
    undecided = []
    for label, key_a, key_b in build_posterior_contrasts(ROSTER_MODELS, ROSTER_INFOS):
        if key_a not in correct or key_b not in correct:
            tally["SKIPPED"] += 1
            continue
        a, b, seed_idx = paired.aligned(correct, valid, key_a, key_b, drop_invalid=False)
        n_seeds = int(np.unique(seed_idx).size)
        p = paired.cmh_unpaired_p(a, b, seed_idx)
        ci = paired_diff_ci(a, b, seed_idx, alpha=2 * ALPHA_USED)
        state = classify(p, ci["lo"], ci["hi"], MEI, n_seeds, ALPHA_USED)
        tally[state] += 1
        if state == "UNDECIDED":
            undecided.append((label, key_a, key_b, n_seeds))
        print(f"{label:>46} {ci['diff']:>+8.4f} "
              f"[{ci['lo']:>+8.4f},{ci['hi']:>+8.4f}] {p:>10.2e}  {state}")

    print("-" * 104)
    print(f"DECIDED {tally['DECIDED']}   EQUIVALENT {tally['EQUIVALENT']}   "
          f"UNDECIDED {tally['UNDECIDED']}   SKIPPED {tally['SKIPPED']}")
    print(f"invalid marks (score: null), counted as failures above: "
          f"{sum(invalid_counts.values())} over {len(invalid_counts)} conditions")

    if not undecided:
        print(f"\nNOTHING FURTHER NEEDED at MEI={MEI:.3f}: every contrast is "
              "resolved or demonstrated equivalent.")
    else:
        # Sizing runs at the MEI, not at the observed effect. Sizing from an
        # observed effect is biased: the same noise that made the effect look
        # large is what selected it into this list.
        # COST NOTE: at this alpha a contrast that never reaches 80% power
        # scans all MAX_REPLICATES values x N_SIMS simulations, twice per
        # contrast. Expect minutes per undecided contrast.
        rng = np.random.default_rng(power_common.SEED)
        print(f"\n{len(undecided)} undecided -- replicates needed "
              f"(R at MEI is the honest target; R at observed is context only):")
        print(f"\n{'contrast':>46} {'R now':>6} {'R for MEI':>10} {'R at observed':>14}")
        for label, key_a, key_b, n_seeds in undecided:
            base = np.array([np.mean([correct[key_a][s][k] for s in correct[key_a]])
                             for k in range(ind_pa.N_HARMONICS)])
            other = np.array([np.mean([correct[key_b][s][k] for s in correct[key_b]])
                              for k in range(ind_pa.N_HARMONICS)])
            shifted = np.clip(base - MEI, 0.0, 1.0)
            # replicates_needed returns (needed, curve) over POWER_TARGETS.
            need_mei, _ = ind_pa.replicates_needed(base, shifted, rng, alpha=ALPHA_USED)
            need_obs, _ = ind_pa.replicates_needed(base, other, rng, alpha=ALPHA_USED)
            target = power_common.POWER_TARGETS[0]      # 0.80
            print(f"{label:>46} {n_seeds:>6} "
                  f"{ind_pa.fmt_r(need_mei[target], ind_pa.MAX_REPLICATES):>10} "
                  f"{ind_pa.fmt_r(need_obs[target], ind_pa.MAX_REPLICATES):>14}")
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

## Section 8 -- score-level flip rate (DETERMINISM_PLAN section 6.2)

**What this is.** The study's direct measurement of cross-**process**
generation nondeterminism. 200 Mathlib-only measurable cells of the
`nemotron-3-nano-4b` deduction lane were re-generated on a fresh box, and
both legs were graded by **today's** verifier, so verifier drift cannot leak
into the comparison (a separate `verifier_drift_stats` isolates that
instead). The estimator is a McNemar-style 2x2 of
re-verified-original vs rerun pass@1, reporting `b + c` discordant cells, the
flip rate, an **exact Clopper-Pearson** interval on it, and the implied
normal-approximation SE on pass@1.

That SE and the CP interval both assume independent cells. Several cells in a
200-cell sample can share a theorem, and a theorem-level effect would
correlate their flips -- so `flip_stats` carries the caveat *inside the JSON
report*, and this notebook prints it verbatim rather than quoting the number
alone.

**Inputs.** The archived report
`notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/flip_report.json`,
streamed from S3. The originals leg
(`originals_rerun/`) is **never** touched and never leaves the machine it was
graded on, by that script's own hard rule.

**What was ported.** The pure estimators only:
`clopper_pearson_interval` (with `_binom_cdf`/`_bisect_decreasing`),
`flip_stats`, `verifier_drift_stats`, `is_pass`,
`dedupe_rows_earliest_wins`, `measurable_cell_keys`, `select_sample_keys`.
Everything else in that 1748-line script is orchestration (EC2, spooling,
Lean verification, stage plumbing) -- see the closing section.

**Descends from.** `scripts/flip_probe.py --stage analyze`, archived in
**`pr4_scripts_2026-08-25.zip`**, and the archived record
`notebooks/DETERMINISM_PLAN_2026-08-16.md` section 6.2
(`pr4_notebook_records_2026-08-25.zip`).

In [ ]:
"""The pure estimators from flip_probe.py --stage analyze."""
import math
import random
from collections.abc import Iterable, Mapping, Sequence

from smolbench.deduction.lean import runner

#: Theorem ``file_path`` marker for a Std (not Mathlib) theorem.
_STD_PACKAGE_MARKER = ".lake/packages/std/"

#: A cell counts as measurable only on a POSITIVE verdict whitelist. Anything
#: else is a VERIFIER outcome ("exception", "replay_failed") or the ungraded
#: sentinel, not a judgment on the candidate proof.
MEASURABLE_VERDICTS = ("success", "lean_error", "incomplete", "given_up")


def dedupe_rows_earliest_wins(rows: Iterable[dict]) -> dict[tuple, dict]:
    """Collapse rows to one per cell key, keeping the FIRST occurrence."""
    by_key: dict[tuple, dict] = {}
    for row in rows:
        if row.get("kind") != "cell":
            continue
        key = runner._row_key(
            row.get("model", ""), row.get("theorem_id", ""),
            int(row.get("k", -1)), row.get("rung", ""),
            int(row.get("replicate_idx", -1)),
        )
        if key not in by_key:
            by_key[key] = row
    return by_key


def is_mathlib_cell(row: dict) -> bool:
    """Check that a cell's theorem is vendored from Mathlib, not Std.

    A missing ``file_path`` is not evidence of being a Std theorem, so it
    is treated as Mathlib.
    """
    return _STD_PACKAGE_MARKER not in str(row.get("file_path") or "")


def measurable_cell_keys(rows: Iterable[dict]) -> list[tuple]:
    """Sorted Mathlib-only cell keys whose surviving verdict is measurable.

    Deduplication runs BEFORE the verdict and Mathlib filters, so a cell
    with several surviving rows contributes at most once. The ascending
    sort is what makes `select_sample_keys` reproducible independent of
    the rows' on-disk order.
    """
    by_key = dedupe_rows_earliest_wins(rows)
    return sorted(
        key for key, row in by_key.items()
        if row.get("verdict") in MEASURABLE_VERDICTS and is_mathlib_cell(row)
    )


def select_sample_keys(measurable: Sequence[tuple], n: int, seed: int) -> list[tuple]:
    """Draw a reproducible n-cell sample from an ALREADY-SORTED population."""
    return random.Random(seed).sample(list(measurable), n)


def _binom_cdf(k: int, n: int, p: float) -> float:
    """``P(X <= k)`` for ``X ~ Binomial(n, p)``, in pure stdlib."""
    if p <= 0.0:
        return 1.0
    if p >= 1.0:
        return 1.0 if k >= n else 0.0
    return sum(math.comb(n, i) * p ** i * (1 - p) ** (n - i) for i in range(k + 1))


def _bisect_decreasing(f, lo: float, hi: float, iters: int = 100) -> float:
    """Root of a MONOTONE DECREASING `f` on ``[lo, hi]``, by bisection."""
    for _ in range(iters):
        mid = (lo + hi) / 2
        if f(mid) > 0:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


def clopper_pearson_interval(k: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    """Exact binomial (Clopper-Pearson) interval, by bisection on the CDF.

    ``lower == 0.0`` iff ``k == 0`` and ``upper == 1.0`` iff ``k == n``:
    the standard boundary convention, since there is no informative bound
    to solve for at either extreme. Bisection (not ``scipy.stats.beta.ppf``)
    keeps the numbers identical on ``.venv`` and ``.venv-lean``.
    """
    if n <= 0:
        raise ValueError(f"clopper_pearson_interval: n must be positive, got {n}")
    if not 0 <= k <= n:
        raise ValueError(f"clopper_pearson_interval: k must be in [0, {n}], got {k}")
    lower = (0.0 if k == 0 else
             _bisect_decreasing(lambda p: _binom_cdf(k - 1, n, p) - (1 - alpha / 2), 0.0, 1.0))
    upper = (1.0 if k == n else
             _bisect_decreasing(lambda p: _binom_cdf(k, n, p) - alpha / 2, 0.0, 1.0))
    return lower, upper


def is_pass(verdict: str) -> bool:
    """``True`` iff `verdict` is exactly ``"success"``.

    Raises on the generation-time sentinel ``"unverified"``, deliberately
    rather than returning False: scoring "never measured" as "measured and
    lost" biases every paired b/c statistic downward, invisibly.
    """
    if verdict == "unverified":
        raise ValueError(
            'is_pass: verdict is "unverified" -- the generation-time sentinel '
            "for an ungraded row, not a graded outcome. Filter ungraded rows "
            "out before calling is_pass (see measurable_cell_keys)."
        )
    return verdict == "success"


PASS_AT_1_SE_CAVEAT = (
    "Normal-approximation SE (and the Clopper-Pearson CI) assume "
    "independent cells. Several cells in this sample can share a "
    "theorem (different rungs/replicates of it), which this "
    "estimate does not account for -- see flip_stats' docstring "
    "Notes. Treat as a rough, likely-too-narrow bound, not exact."
)


def flip_stats(pairs: Mapping[tuple, tuple[str, str]]) -> dict:
    """McNemar-style flip statistics: re-verified original vs rerun verdict.

    `pairs` maps a cell key to the two verdict TEXTS, never pre-computed
    booleans, so a caller cannot silently apply a different pass rule to
    the two legs: this function applies `is_pass` to both itself.
    """
    n = len(pairs)
    a = b = c = d = 0
    flipped_keys: list[tuple] = []
    for key, (orig_verdict, rerun_verdict) in pairs.items():
        orig, rerun = is_pass(orig_verdict), is_pass(rerun_verdict)
        if orig and rerun:
            a += 1
        elif orig and not rerun:
            b += 1
            flipped_keys.append(key)
        elif not orig and rerun:
            c += 1
            flipped_keys.append(key)
        else:
            d += 1
    discordant = b + c
    flip_rate = discordant / n if n else 0.0
    ci_lo, ci_hi = clopper_pearson_interval(discordant, n) if n else (0.0, 0.0)
    se = math.sqrt(flip_rate * (1 - flip_rate) / n) if n else 0.0
    return {
        "n": n,
        "a_both_pass": a,
        "b_orig_pass_rerun_fail": b,
        "c_orig_fail_rerun_pass": c,
        "d_both_fail": d,
        "discordant": discordant,
        "flip_rate": flip_rate,
        "flip_rate_ci95": [ci_lo, ci_hi],
        "pass_at_1_se": se,
        "pass_at_1_se_caveat": PASS_AT_1_SE_CAVEAT,
        "flipped_keys": [list(k) for k in flipped_keys],
    }


def verifier_drift_stats(pairs: Mapping[tuple, tuple[str, str]]) -> dict:
    """Agreement of a fresh reverification against the study's stored verdict.

    Compares the two verdict STRINGS for equality rather than collapsing
    them to pass/fail: drift between e.g. ``lean_error`` and ``incomplete``
    is informative even though neither is a success.
    """
    n = len(pairs)
    agree = 0
    disagreements: list[dict] = []
    for key, (study_verdict, reverified_verdict) in pairs.items():
        if study_verdict == reverified_verdict:
            agree += 1
        else:
            disagreements.append({"key": list(key), "study_verdict": study_verdict,
                                  "reverified_verdict": reverified_verdict})
    return {"n": n, "agree": agree, "agreement_rate": agree / n if n else 0.0,
            "disagreements": disagreements}


print("ported estimators:", ", ".join(sorted(
    ("clopper_pearson_interval", "flip_stats", "verifier_drift_stats", "is_pass",
     "dedupe_rows_earliest_wins", "measurable_cell_keys", "select_sample_keys"))))

In [ ]:
"""Re-render section 6.2's flip table from the archive, and ASSERT it matches."""
FLIP_REPORT = ("notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/"
               "flip_report.json")
SAMPLE_MANIFEST = ("notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/"
                   "sample_manifest.json")

report = archive.json(FLIP_REPORT)
manifest = archive.json(SAMPLE_MANIFEST)
stored = report["flip_stats"]

# Rebuild the pairing the stored 2x2 describes, one DISTINCT key per cell, and
# push it back through the ported flip_stats. This exercises is_pass on real
# verdict strings rather than trusting the stored arithmetic.
pairs = {}
for i in range(stored["a_both_pass"]):
    pairs[("rebuilt", "a", i, "-", 0)] = ("success", "success")
for i in range(stored["b_orig_pass_rerun_fail"]):
    pairs[("rebuilt", "b", i, "-", 0)] = ("success", "lean_error")
for i in range(stored["c_orig_fail_rerun_pass"]):
    pairs[("rebuilt", "c", i, "-", 0)] = ("lean_error", "success")
for i in range(stored["d_both_fail"]):
    pairs[("rebuilt", "d", i, "-", 0)] = ("lean_error", "lean_error")
assert len(pairs) == stored["n"], (len(pairs), stored["n"])
rendered = flip_stats(pairs)

print(f"lane {report['model']}   study run {report['study_run']}   "
      f"flip run {report['flip_run']}")
print(f"sample: {report['n_paired']}/{report['n_requested']} paired, drawn from "
      f"{report['sample_n_measurable_mathlib_population']} measurable Mathlib "
      f"cells (whitelist sha256 {report['sample_whitelist_sha256'][:16]})")
print(f"manifest agrees on the population: "
      f"{manifest.get('n_measurable_mathlib_population')}")
print(f"generated {report['generated_at_utc']}")

print(f"\n2x2 (re-verified original x rerun), n = {rendered['n']}")
print(f"{'':>18} {'rerun pass':>12} {'rerun fail':>12}")
print(f"{'orig pass':>18} {rendered['a_both_pass']:>12} "
      f"{rendered['b_orig_pass_rerun_fail']:>12}")
print(f"{'orig fail':>18} {rendered['c_orig_fail_rerun_pass']:>12} "
      f"{rendered['d_both_fail']:>12}")
print(f"\ndiscordant (b + c) : {rendered['discordant']}")
print(f"flip rate          : {rendered['flip_rate']:.4f}")
print(f"95% Clopper-Pearson: [{rendered['flip_rate_ci95'][0]:.6f}, "
      f"{rendered['flip_rate_ci95'][1]:.6f}]")
print(f"implied pass@1 SE  : {rendered['pass_at_1_se']:.6f}")
print(f"\nCAVEAT: {rendered['pass_at_1_se_caveat']}")

drift = report["verifier_drift"]
print(f"\nverifier drift (same text, today's verifier vs the study's stored "
      f"verdict): {drift['agree']}/{drift['n']} = {drift['agreement_rate']:.4f}")

# The asserts. Every expected value is READ from the archived JSON.
# a/b/c/d are the counts the pairing above was RECONSTRUCTED from, so they
# check the reconstruction; the statistics DERIVED from them -- discordant,
# flip_rate, the Clopper-Pearson interval, the SE, and the caveat string --
# are what the ported estimator actually re-computes here.
for field in ("n", "a_both_pass", "b_orig_pass_rerun_fail",
              "c_orig_fail_rerun_pass", "d_both_fail", "discordant",
              "flip_rate", "pass_at_1_se", "pass_at_1_se_caveat"):
    assert rendered[field] == stored[field], (field, rendered[field], stored[field])
assert rendered["flip_rate_ci95"] == stored["flip_rate_ci95"], (
    rendered["flip_rate_ci95"], stored["flip_rate_ci95"])
# The drift table is a pure recount, so it re-renders exactly too.
drift_pairs = {("rebuilt", "agree", i, "-", 0): ("lean_error", "lean_error")
               for i in range(drift["agree"])}
drift_pairs.update({("rebuilt", "disagree", i, "-", 0):
                    (d["study_verdict"], d["reverified_verdict"])
                    for i, d in enumerate(drift["disagreements"])})
rendered_drift = verifier_drift_stats(drift_pairs)
for field in ("n", "agree", "agreement_rate"):
    assert rendered_drift[field] == drift[field], (field, rendered_drift[field], drift[field])

print("\nASSERT OK: discordant, flip rate, Clopper-Pearson interval, pass@1 SE "
      "and the\n           caveat string all re-derive to the archived record "
      f"({FLIP_REPORT});\n           the 2x2 counts the pairing was rebuilt "
      "from check out too.")

## Section 9 -- the free flip bound (DETERMINISM_PLAN section 6.3)

**What this is.** A zero-cost bound on section 8's flip rate. The 2026-08-15
resampling bug left a handful of deduction cells with **more than one
surviving generation attempt** -- attempts that reached the model and
returned, drawn by different serving processes. Those are already-collected
paired draws of the same cell across processes, so they bound the flip rate
for free. Both attempts of every pair were graded with the same (today's)
verifier, so verifier identity cancels within a pair.

> **The caveat that must ride every use of this number.** The sample is
> **selected on cell outcome**: a cell was re-drawn precisely because its
> first attempt looked empty or failed. So this is not an unbiased estimate.
> Because the sample conditions on the first draw, regression to the mean on
> the second argues the bound *over*-estimates the population flip rate --
> a direction that is **reasoned, not measured**. It is a sanity check on
> section 8's design, **never the headline**. Note in the numbers below that
> all 74 pairs have an empty first attempt and none has two non-empty
> attempts: that is the selection, visible.

**Inputs.** The archived report
`notebooks/deduction/results/flip_free_bound_2026-08-18.json`, streamed from
S3. Everything below is recomputed from its per-pair records
(`report["lanes"][lane]["pairs"]`), not read off its summary.

**The interval is recomputed with section 8's `clopper_pearson_interval`,**
not with the archived script's own `exact_binom_ci`. That function integrated
the beta density on a 200,000-point grid to avoid a scipy dependency; the
bisection estimator is exact to machine precision, and using one interval
estimator across both sections is worth more than keeping a second
implementation alive. The two agree to the 4 decimals the record stores.

**Descends from.** `scripts/flip_free_bound.py`, archived in
**`pr4_scripts_2026-08-25.zip`**, and the archived record
`notebooks/DETERMINISM_PLAN_2026-08-16.md` section 6.3
(`pr4_notebook_records_2026-08-25.zip`).

In [ ]:
"""Recompute the section 6.3 bound from the archived per-pair records."""
FREE_BOUND = "notebooks/deduction/results/flip_free_bound_2026-08-18.json"
free = archive.json(FREE_BOUND)
stored_summary = free["summary"]

# Recompute from the pairs, never from the summary.
all_pairs = [p for lane in free["lanes"].values() for p in lane["pairs"]]
n = len(all_pairs)


def _flip(pair, last: bool = False) -> bool:
    """Re-derive a pair's flip from its VERDICTS, via section 8's pass rule."""
    passes = [is_pass(v) for v in pair["verdicts"]]
    return passes[0] != (passes[-1] if last else passes[1])


# Re-derive rather than re-sum the stored boolean, then pin the two together:
# the stored flag is a summary of these verdicts and must agree with them.
assert all(_flip(p) == p["flip_first_vs_second"] for p in all_pairs)
assert all(_flip(p, last=True) == p["flip_first_vs_last"] for p in all_pairs)
flips = sum(1 for p in all_pairs if _flip(p))
flips_last = sum(1 for p in all_pairs if _flip(p, last=True))
identical = sum(1 for p in all_pairs if p["identical_text"])
std_pairs = sum(1 for p in all_pairs if p["is_std"])
first_empty = sum(1 for p in all_pairs if p["first_empty"])
two_nonempty = sum(1 for p in all_pairs if p["n_nonempty_attempts"] >= 2)

mathlib = [p for p in all_pairs if not p["is_std"]]
n_math = len(mathlib)
flips_math = sum(1 for p in mathlib if _flip(p))

rate = flips / n
lo, hi = clopper_pearson_interval(flips, n)
lo_m, hi_m = clopper_pearson_interval(flips_math, n_math)

print("per-lane pairs (cells with >= 2 surviving attempts):")
for lane, rec_lane in free["lanes"].items():
    lane_pairs = rec_lane["pairs"]
    lane_flips = sum(1 for p in lane_pairs if _flip(p))
    print(f"  {lane:>16}: {len(lane_pairs):>3} pairs "
          f"(inventory expected {rec_lane['inventory_expected']}), "
          f"{lane_flips} flip(s)")

print(f"\nALL pairs        : {flips}/{n} = {rate:.4f}   "
      f"95% CP [{lo:.4f}, {hi:.4f}]")
print(f"Mathlib-only     : {flips_math}/{n_math} = {flips_math / n_math:.4f}   "
      f"95% CP [{lo_m:.4f}, {hi_m:.4f}]   (Std pairs excluded: {std_pairs})")
print(f"first-vs-LAST    : {flips_last}/{n}")
print(f"identical text   : {identical}/{n}")
print(f"first attempt empty          : {first_empty}/{n}")
print(f"pairs with two non-empty text: {two_nonempty}/{n}")

print(f"\nCAVEAT (carried in the record itself): {free['caveat']}")
print("This bound is a sanity check on section 6.2's design. It is NEVER the "
      "headline:\nthe headline flip rate is section 8's "
      f"{archive.json(FLIP_REPORT)['flip_stats']['flip_rate']:.4f} on a "
      "randomly drawn sample.")

# Asserts against the stored headline -- all expected values READ from the JSON.
assert n == stored_summary["n_pairs"], (n, stored_summary["n_pairs"])
assert flips == stored_summary["flips_first_vs_second"], (
    flips, stored_summary["flips_first_vs_second"])
assert round(rate, 4) == stored_summary["flip_rate"], (
    round(rate, 4), stored_summary["flip_rate"])
assert [round(lo, 4), round(hi, 4)] == stored_summary["ci95"], (
    [round(lo, 4), round(hi, 4)], stored_summary["ci95"])
assert flips_last == stored_summary["flips_first_vs_last"]
assert identical == stored_summary["identical_text_pairs"]
assert std_pairs == stored_summary["std_pairs"]
assert first_empty == stored_summary["pairs_first_attempt_empty"]
assert two_nonempty == stored_summary["pairs_with_two_nonempty_attempts"]
# The Mathlib subset: the record stores the COUNTS but no interval, so the
# counts are asserted and the interval above is reported as derived.
assert flips_math == stored_summary["flips_mathlib_only"], (
    flips_math, stored_summary["flips_mathlib_only"])
assert n_math == stored_summary["n_mathlib_pairs"], (n_math, stored_summary["n_mathlib_pairs"])

print(f"\nASSERT OK: recomputed headline {flips}/{n} and its CI equal the "
      f"archived record ({FREE_BOUND});")
print(f"           Mathlib-only subset {flips_math}/{n_math} equals it too "
      "(its CI is derived here, not stored).")

## What was deliberately left out of the three ports

Sections 7-9 port **statistics**. The rest of those three scripts is
machinery, and machinery does not belong in an analysis notebook. Left out,
by category:

**Orchestration and I/O** -- `posterior_power.load_study` /
`invalid_counts` (regex loaders superseded by `paired_analysis.load_marks`,
which also returns the validity mask); `flip_free_bound._rec`,
`stream_all_rows`, `surviving`, `verify_rows_in_place` and its Lean/Dojo lock
(a GPU/Lean grading pass, not a statistic); `flip_probe`'s entire stage
machinery (`_stage_sample` / `_generate` / `_verify` / `_analyze`,
`_spool_flip_run`, `_LocalRunClient`, `_LocalBody`, `_prepare_originals_rows`,
`_read_flip_run_verified_rows`, the `_flip_run_dir` / `_whitelist_path` /
`_originals_dir` path helpers, `stock_vllm_args`, and its EC2 constants).

**Superseded numerics** (three statistics that *were* ported, but onto the
live modules instead of as copies) -- `posterior_power.cmh` +
`chi2_sf_1df` -> `paired_analysis.cmh_unpaired_p`; `posterior_power.boot_ci`
+ `_ladder` + its local `replicates_needed` ->
`error_bars.bootstrap_stats`/`diff_ci` and
`power_analysis.replicates_needed`; `flip_free_bound.exact_binom_ci` (a
200k-point grid integration of the beta density) ->
`clopper_pearson_interval`.

**Gates, not statistics** -- `flip_probe.reject_unverified_rows` and
`assert_population_size` are refusals that guard a *collection* run: they
protect the sample as it is drawn and verified, and there is no sample being
drawn here. `is_pass`'s own refusal on `"unverified"` **is** ported, because
that one guards the estimator itself.

**Re-running the ports.** Sections 8 and 9 read the archive and assert
against it, so they run on any machine with credentials. Section 7's
classifier is exercised on synthetic counts in the same way; its run over the
real block needs the results store and is gated with everything else.